# Notebook 04 — Post-hoc Probability Calibration (Platt + Isotonic)
### Fit on calibration split, applied to test — before/after Brier/ECE/LogLoss

In [1]:
import os
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.npy'):
            print(os.path.join(root, f))

/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_maven_LogReg_natural_cal.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_cell_XGB_smote_test.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_maven_XGB_natural_cal.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_maven_RF_natural_cal.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_maven_LGBM_classweight_test.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_cell_LGBM_classweight_test.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_cell_LGBM_natural_cal.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_maven_RF_classweight_test.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_cell_RF_natural_test.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_maven_LogReg_smote_test.npy
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3/prob_cell_LogReg_classweight_test.npy
/kaggle/i

In [2]:

# ===== CELL 1: imports + config =====
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
 
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
 
PROB_DIR  = "/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-3"   # NB03 probs
SPLIT_DIR = "/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1"   # NB01 splits
OUT_DIR   = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
 
DATASETS   = ["maven", "cell"]
MODELS     = ["LogReg", "RF", "XGB", "LGBM"]
STRATEGIES = ["natural", "classweight", "smote"]
EPS = 1e-6
 

In [3]:

# ===== CELL 2: helpers =====
def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    ece, n = 0.0, len(y_true)
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            ece += (m.sum() / n) * abs(y_true[m].mean() - y_prob[m].mean())
    return ece
 
def _logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))
 
def platt_fit_apply(p_cal, y_cal, p_target):
    """Platt scaling: logistic regression on logit(prob), fit on cal, apply to target."""
    lr = LogisticRegression()
    lr.fit(_logit(p_cal).reshape(-1, 1), y_cal)
    return lr.predict_proba(_logit(p_target).reshape(-1, 1))[:, 1]
 
def isotonic_fit_apply(p_cal, y_cal, p_target):
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
    iso.fit(p_cal, y_cal)
    return iso.predict(p_target)
 
def metrics(y, p):
    return {
        "AUC": roc_auc_score(y, p),
        "Brier": brier_score_loss(y, p),
        "LogLoss": log_loss(y, np.clip(p, EPS, 1 - EPS), labels=[0, 1]),
        "ECE": ece_score(y, p),
    }
 
def load_y(ds):
    y_cal  = pd.read_csv(f"{SPLIT_DIR}/{ds}_cal.csv")["target"].values
    y_test = pd.read_csv(f"{SPLIT_DIR}/{ds}_test.csv")["target"].values
    return y_cal, y_test
 
 

In [4]:

# ===== CELL 3: run calibration for every combo =====
rows = []
for ds in DATASETS:
    y_cal, y_test = load_y(ds)
    for model in MODELS:
        for strat in STRATEGIES:
            base = f"{ds}_{model}_{strat}"
            p_cal  = np.load(f"{PROB_DIR}/prob_{base}_cal.npy")
            p_test = np.load(f"{PROB_DIR}/prob_{base}_test.npy")
 
            # --- uncalibrated (none) ---
            m = metrics(y_test, p_test); m.update(dict(dataset=ds, model=model,
                                                       strategy=strat, calib="none"))
            rows.append(m)
            np.save(f"{OUT_DIR}/cprob_{base}_none_cal.npy",  p_cal)
            np.save(f"{OUT_DIR}/cprob_{base}_none_test.npy", p_test)
 
            # --- Platt ---
            pc = platt_fit_apply(p_cal, y_cal, p_cal)     # calibrated cal (for NB05 thresholding)
            pt = platt_fit_apply(p_cal, y_cal, p_test)    # calibrated test
            m = metrics(y_test, pt); m.update(dict(dataset=ds, model=model,
                                                   strategy=strat, calib="platt"))
            rows.append(m)
            np.save(f"{OUT_DIR}/cprob_{base}_platt_cal.npy",  pc)
            np.save(f"{OUT_DIR}/cprob_{base}_platt_test.npy", pt)
 
            # --- Isotonic ---
            pc = isotonic_fit_apply(p_cal, y_cal, p_cal)
            pt = isotonic_fit_apply(p_cal, y_cal, p_test)
            m = metrics(y_test, pt); m.update(dict(dataset=ds, model=model,
                                                   strategy=strat, calib="isotonic"))
            rows.append(m)
            np.save(f"{OUT_DIR}/cprob_{base}_isotonic_cal.npy",  pc)
            np.save(f"{OUT_DIR}/cprob_{base}_isotonic_test.npy", pt)
 
    print(f"{ds}: done")
 

maven: done
cell: done


In [5]:

# ===== CELL 4: full results table =====
res = pd.DataFrame(rows)[
    ["dataset", "model", "strategy", "calib", "AUC", "Brier", "LogLoss", "ECE"]
].round(4)
res.to_csv(f"{OUT_DIR}/results_calibration.csv", index=False)
print("\n============== CALIBRATION RESULTS (test) ==============")
print(res.to_string(index=False))
 


============== CALIBRATION RESULTS (test) ==============
dataset  model    strategy    calib    AUC  Brier  LogLoss    ECE
  maven LogReg     natural     none 0.9043 0.1097   0.3490 0.0309
  maven LogReg     natural    platt 0.9043 0.1099   0.3541 0.0320
  maven LogReg     natural isotonic 0.9034 0.1123   0.3579 0.0398
  maven LogReg classweight     none 0.9039 0.1287   0.3937 0.0999
  maven LogReg classweight    platt 0.9039 0.1104   0.3550 0.0443
  maven LogReg classweight isotonic 0.9022 0.1119   0.3590 0.0377
  maven LogReg       smote     none 0.9034 0.1267   0.3894 0.0896
  maven LogReg       smote    platt 0.9034 0.1105   0.3555 0.0401
  maven LogReg       smote isotonic 0.9011 0.1118   0.3657 0.0393
  maven     RF     natural     none 0.9190 0.0988   0.3148 0.0232
  maven     RF     natural    platt 0.9190 0.0986   0.3127 0.0222
  maven     RF     natural isotonic 0.9169 0.1016   0.3534 0.0377
  maven     RF classweight     none 0.9198 0.0992   0.3154 0.0260
  maven     RF cla

In [6]:

# ===== CELL 5: before/after summary — does calibration fix ECE/Brier? =====
piv_ece = (res.pivot_table(index=["dataset", "strategy"], columns="calib",
                           values="ECE", aggfunc="mean")
              [["none", "platt", "isotonic"]].round(4))
piv_brier = (res.pivot_table(index=["dataset", "strategy"], columns="calib",
                             values="Brier", aggfunc="mean")
                [["none", "platt", "isotonic"]].round(4))
print("\n===== mean ECE by strategy (none -> platt -> isotonic) =====")
print(piv_ece.to_string())
print("\n===== mean Brier by strategy (none -> platt -> isotonic) =====")
print(piv_brier.to_string())
piv_ece.to_csv(f"{OUT_DIR}/calib_ece_summary.csv")
piv_brier.to_csv(f"{OUT_DIR}/calib_brier_summary.csv")
 


===== mean ECE by strategy (none -> platt -> isotonic) =====
calib                  none   platt  isotonic
dataset strategy                             
cell    classweight  0.1366  0.0134    0.0103
        natural      0.0130  0.0114    0.0095
        smote        0.0610  0.0120    0.0108
maven   classweight  0.0609  0.0288    0.0271
        natural      0.0366  0.0257    0.0300
        smote        0.0550  0.0284    0.0333

===== mean Brier by strategy (none -> platt -> isotonic) =====
calib                  none   platt  isotonic
dataset strategy                             
cell    classweight  0.2147  0.1904    0.1906
        natural      0.1902  0.1902    0.1903
        smote        0.2009  0.1907    0.1908
maven   classweight  0.1081  0.1000    0.1014
        natural      0.1006  0.0992    0.1007
        smote        0.1058  0.1000    0.1016


In [7]:

# ===== CELL 6: AUC invariance sanity check =====
auc_check = (res.pivot_table(index=["dataset", "model", "strategy"], columns="calib",
                             values="AUC", aggfunc="mean"))
auc_check["max_shift"] = (auc_check[["none", "platt", "isotonic"]].max(axis=1)
                          - auc_check[["none", "platt", "isotonic"]].min(axis=1))
print("\n===== max AUC shift from calibration (should be ~0) =====")
print(f"largest AUC shift across all combos: {auc_check['max_shift'].max():.5f}")
 
# ============================================================================
# DONE. Saved calibrated cal+test probs (cprob_*.npy) for Notebook 05.
# Next: Notebook 05 — CLV + cost/profit framework, EMPC-style profit-maximizing
#       threshold tuned on CAL, realized profit on TEST; compare uncalibrated vs
#       calibrated, and F1-threshold vs profit-threshold. The central result.
# ============================================================================
 


===== max AUC shift from calibration (should be ~0) =====
largest AUC shift across all combos: 0.00260
